# 12. Put the chosen model on the phone

Run this after notebook 10.

- **If the chosen model is Small CNN:** you have nothing to do. The app already runs it offline on the phone.
- **If it is MobileNetV3, EfficientNet-B0 or ConvNeXt-Tiny:** this notebook converts it to ONNX, a format the phone's browser can run. It then checks that the converted model gives the same answers, and copies everything into the app.

It installs PyTorch, which is a large download of about 200 MB. A GPU isn't needed.

In [ ]:
%pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cpu

In [ ]:
%pip install -q onnx onnxruntime onnxscript numpy pandas pillow

In [ ]:
from pathlib import Path

# Finds the project folder wherever this notebook is opened from.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw" / "DeepDRiD").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find the AfriEyeSamsung project folder. Open this notebook from inside it.")
PROCESSED, RESULTS, MODELS = ROOT / "data" / "processed", ROOT / "results", ROOT / "models"
print("Project folder:", ROOT)

In [ ]:
import json
import shutil
import urllib.request

import numpy as np

choice = json.loads((RESULTS / "final_choice.json").read_text())
SLUG = choice["slug"]
print("Chosen model:", choice["model"])
NEEDS_EXPORT = SLUG != "small_cnn"
if not NEEDS_EXPORT:
    print("Nothing to export. The app already uses this model.")

In [ ]:
if NEEDS_EXPORT:
    import torch
    import torch.nn as nn
    import torchvision as tv

    def build(slug):
        if slug == "mobilenet_v3":
            m = tv.models.mobilenet_v3_large(); m.classifier[3] = nn.Linear(m.classifier[3].in_features, 5)
        elif slug == "efficientnet_b0":
            m = tv.models.efficientnet_b0(); m.classifier[1] = nn.Linear(m.classifier[1].in_features, 5)
        elif slug == "convnext_tiny":
            m = tv.models.convnext_tiny(); m.classifier[2] = nn.Linear(m.classifier[2].in_features, 5)
        else:
            raise ValueError(f"This notebook can't export {slug}.")
        return m

    class WithProbabilities(nn.Module):
        def __init__(self, model):
            super().__init__()
            self.model = model
        def forward(self, x):
            return torch.sigmoid(self.model(x))

    m = metrics = json.loads((RESULTS / f"{SLUG}_metrics.json").read_text())
    SIZE = m["image_size"]
    net = build(SLUG)
    net.load_state_dict(torch.load(MODELS / f"{SLUG}.pt", map_location="cpu"))
    wrapped = WithProbabilities(net).eval()
    target = MODELS / f"{SLUG}.onnx"
    example = torch.zeros(1, 3, SIZE, SIZE)
    try:
        torch.onnx.export(wrapped, example, str(target), input_names=["photo"], output_names=["scores"], opset_version=17, dynamo=False)
    except TypeError:
        torch.onnx.export(wrapped, example, str(target), input_names=["photo"], output_names=["scores"], opset_version=17)
    print(f"Saved {target.name}, {target.stat().st_size / 1e6:.1f} MB")

## Check the converted model gives the same answers

In [ ]:
if NEEDS_EXPORT:
    import onnxruntime as ort

    session = ort.InferenceSession(str(target), providers=["CPUExecutionProvider"])
    x = torch.from_numpy(np.random.default_rng(0).normal(size=(1, 3, SIZE, SIZE)).astype(np.float32))
    with torch.no_grad():
        original = wrapped(x).numpy()
    converted = session.run(None, {session.get_inputs()[0].name: x.numpy()})[0]
    gap = float(np.abs(original - converted).max())
    print("Largest difference:", gap)
    assert gap < 1e-3, "The converted model does not match. Do not use it."

## Copy it into the app

In [ ]:
if NEEDS_EXPORT:
    config = {
        "name": choice["model"], "slug": SLUG, "size": SIZE, "mean": [.485, .456, .406], "std": [.229, .224, .225],
        "thresholds": {"referable": m["threshold"], "urgent": m["urgent_threshold"], "quality": m["quality_threshold"]}, "file": f"{SLUG}.onnx",
    }
    (MODELS / f"{SLUG}_phone.json").write_text(json.dumps(config, indent=1))
    app_model = ROOT / "app" / "static" / "model"
    app_model.mkdir(parents=True, exist_ok=True)
    shutil.copy(target, app_model / target.name)
    (app_model / "model_config.json").write_text(json.dumps(config, indent=1))

    # The phone needs this runtime saved locally, so that it still works without internet.
    version = "1.19.2"
    vendor = ROOT / "app" / "static" / "vendor"
    vendor.mkdir(parents=True, exist_ok=True)
    for name in ("ort.min.js", "ort-wasm-simd-threaded.wasm", "ort-wasm-simd-threaded.mjs"):
        if not (vendor / name).exists():
            urllib.request.urlretrieve(f"https://cdn.jsdelivr.net/npm/onnxruntime-web@{version}/dist/{name}", vendor / name)
    print("The app will use", choice["model"], "the next time it starts.")
    print("Open the CHW screen, scan a sample photo, and check that the result names this model.")